In [ ]:
#Auto-reload edited modules (e.g. src/utils) without restarting the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
#Install dependencies (only the ones that are missing)
from utils import ensure_installed

ensure_installed(["anthropic", "python-dotenv"])

In [ ]:
#Create the Claude client (loads .env)
from utils import ClaudeClient

claude = ClaudeClient( max_tokens=100)
print("Claude model:", claude.model)

In [ ]:
# Functions to validate the output structure
import re
import ast
import json


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    # Haiku does not support assistant-message prefill ("```json" + stop sequence),
    # so ask for structured output with a JSON schema instead.
    dataset_schema = {
        "type": "object",
        "properties": {
            "tasks": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "task": {"type": "string"},
                        "format": {"type": "string", "enum": ["json", "python", "regex"]},
                        "solution_criteria": {"type": "string"}
                    },
                    "required": ["task", "format", "solution_criteria"],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["tasks"],
        "additionalProperties": False,
    }
    messages = []
    claude.add_user_message(messages, prompt)
    text = claude.chat(
        messages,
        max_tokens=1000,
        output_config={"format": {"type": "json_schema", "schema": dataset_schema}},
    )
    # The schema wraps the list in {"tasks": [...]}; return just the list of test cases
    print(text)
    return json.loads(text)

In [ ]:
from pathlib import Path

dataset = generate_dataset()
print(dataset)
# Project root = first parent folder that contains pyproject.toml
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

output_path = PROJECT_ROOT / "data" / "code-eval-dataset_v1.json"
output_path.parent.mkdir(parents=True, exist_ok=True)   # create the folder if it doesn't exist

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print(f"Saved {len(dataset)} items to {output_path}")

In [ ]:
with open(output_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)
print(dataset)
print(f"Loaded {len(dataset)} items from {output_path}")

In [ ]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    solution_schema = {
        "type": "object",
        "properties": {
            "code": {"type": "string"},
        },
        "required": ["code"],
        "additionalProperties": False,
    }
    messages = []
    claude.add_user_message(messages, prompt)

    text = claude.chat(
        messages,
        max_tokens=1000,
        output_config={"format": {"type": "json_schema", "schema": solution_schema}},
    )
    # Return only the raw code, so grade_syntax checks the code itself
    # rather than the {"code": ...} JSON wrapper
    return json.loads(text)["code"]

In [ ]:
grade_schema = {
    "type": "object",
    "properties": {
        "strengths": {"type": "array", "items": {"type": "string"}},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
        "reasoning": {"type": "string"},
        "score": {"type": "integer"},
    },
    "required": ["strengths", "weaknesses", "reasoning", "score"],
    "additionalProperties": False,
}


def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

`   Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Criteria you should use to evaluate the solution:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>

    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}`
    """
    messages = []
    claude.add_user_message(messages, eval_prompt)
    # This model doesn't support assistant-message prefill ("```json" + stop sequence),
    # so use structured output with a JSON schema instead (same as the dataset cell).
    text = claude.chat(
        messages,
        max_tokens=1000,
        output_config={"format": {"type": "json_schema", "schema": grade_schema}},
    )
    return json.loads(text)

In [ ]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "syntax_score": syntax_score
    }

In [ ]:
from statistics import mean
def run_eval(test_cases):
    results = []
    for test_case in test_cases:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print("Average score:", average_score)
    return results

In [ ]:
with open(output_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

test_cases = dataset["tasks"]

results = run_eval(test_cases)   
print(results)